# Phase 2.4 — Training Dataset & Splits

Create leakage-safe temporal train/validation/test datasets and the final user-item training representation.


In [5]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "data" / "processed").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

INPUT_PATH = PROCESSED_DIR / "interaction_records.csv"
TRAIN_PATH = PROCESSED_DIR / "train_interactions.csv"
VALIDATION_PATH = PROCESSED_DIR / "validation_interactions.csv"
TEST_PATH = PROCESSED_DIR / "test_interactions.csv"
TRAIN_USER_ITEM_PATH = PROCESSED_DIR / "train_user_item.csv"

print("Input:", INPUT_PATH)


Input: f:\annuspeaks.com\recommendation-system\data\processed\interaction_records.csv


## Temporal Split Strategy

Interactions are ordered by timestamp separately for each user.

- Users with at least 3 interactions: latest interaction → test, second-latest → validation, earlier interactions → training.
- Users with fewer than 3 interactions: keep their observed interactions in training so sparse users are not completely removed from model fitting.
- The split is chronological and deterministic.
- Equal timestamps are allowed at a split boundary because Retailrocket can contain multiple events for the same user at the same millisecond; equal timestamps are not treated as future leakage.


In [6]:
# Load the interaction records produced in Phase 2.1.

columns = [
    "user_id",
    "item_id",
    "interaction_type",
    "weight",
    "timestamp",
]

interactions = pd.read_csv(
    INPUT_PATH,
    usecols=columns
)

interactions = interactions.sort_values(
    ["user_id", "timestamp"],
    kind="mergesort"
).reset_index(drop=True)

print("Interactions loaded:", f"{len(interactions):,}")
print("Users:", f"{interactions['user_id'].nunique():,}")


Interactions loaded: 2,756,101
Users: 1,407,580


In [7]:
# Assign chronological positions within each user's history.

interactions["user_interaction_number"] = (
    interactions.groupby("user_id").cumcount()
)

interactions["user_interaction_total"] = (
    interactions.groupby("user_id")["user_id"].transform("size")
)

test_mask = (
    interactions["user_interaction_total"] >= 3
) & (
    interactions["user_interaction_number"]
    == interactions["user_interaction_total"] - 1
)

validation_mask = (
    interactions["user_interaction_total"] >= 3
) & (
    interactions["user_interaction_number"]
    == interactions["user_interaction_total"] - 2
)

test_data = interactions.loc[test_mask, columns].copy()
validation_data = interactions.loc[validation_mask, columns].copy()
train_data = interactions.loc[
    ~(test_mask | validation_mask),
    columns
].copy()

print("Train:", f"{len(train_data):,}")
print("Validation:", f"{len(validation_data):,}")
print("Test:", f"{len(test_data):,}")


Train: 2,356,045
Validation: 200,028
Test: 200,028


## Leakage Check

The maximum training timestamp must be earlier than the validation/test interaction for each user. The split is based only on chronological user history.


In [8]:
# Verify per-user temporal ordering.

train_last = train_data.groupby("user_id")["timestamp"].max()
validation_first = validation_data.groupby("user_id")["timestamp"].min()
test_first = test_data.groupby("user_id")["timestamp"].min()

common_val_users = train_last.index.intersection(validation_first.index)
common_test_users = train_last.index.intersection(test_first.index)

# Equal timestamps are allowed because multiple Retailrocket events can occur
# at the same millisecond. No validation/test event occurs earlier than the
# latest training interaction for the same user.
assert (
    (train_last.loc[common_val_users] <= validation_first.loc[common_val_users])
    .all()
)

assert (
    (train_last.loc[common_test_users] <= test_first.loc[common_test_users])
    .all()
)

print("Temporal leakage check: PASS")
print("Equal timestamp boundaries:", 
      int((train_last.loc[common_val_users] == validation_first.loc[common_val_users]).sum()))


Temporal leakage check: PASS
Equal timestamp boundaries: 317


## Final User-Item Training Representation

Aggregate training interactions by user and product.

This representation keeps the strongest information needed for initial collaborative filtering and later recommendation modeling: interaction count, total behavioral weight, and the latest observed interaction time.


In [9]:
# Build the final user-item training representation.

train_user_item = (
    train_data
    .groupby(["user_id", "item_id"])
    .agg(
        interaction_count=("interaction_type", "size"),
        total_weight=("weight", "sum"),
        last_timestamp=("timestamp", "max"),
    )
    .reset_index()
)

print("User-item training pairs:", f"{len(train_user_item):,}")
display(train_user_item.head())


User-item training pairs: 1,939,777


,user_id,item_id,interaction_count,total_weight,last_timestamp
0,0,285930,1,1,1442004589439
1,1,72028,1,1,1439487966444
2,2,216305,1,1,1438970468920
3,2,259884,1,1,1438970212664
4,2,325215,2,2,1438970013790


In [10]:
# Save reproducible train/validation/test datasets.

for path, data in [
    (TRAIN_PATH, train_data),
    (VALIDATION_PATH, validation_data),
    (TEST_PATH, test_data),
    (TRAIN_USER_ITEM_PATH, train_user_item),
]:
    data.to_csv(path, index=False)
    print(f"Saved {path.name}: {len(data):,} rows")


Saved train_interactions.csv: 2,356,045 rows
Saved validation_interactions.csv: 200,028 rows
Saved test_interactions.csv: 200,028 rows
Saved train_user_item.csv: 1,939,777 rows


In [11]:
# Final Phase 2.4 validation

saved_paths = [
    TRAIN_PATH,
    VALIDATION_PATH,
    TEST_PATH,
    TRAIN_USER_ITEM_PATH,
]

for path in saved_paths:
    assert path.exists(), f"Missing output: {path}"

assert len(train_data) > 0
assert len(validation_data) > 0
assert len(test_data) > 0

assert set(train_data.columns) == set(columns)
assert not train_data.empty
assert set(validation_data.columns) == set(columns)
assert set(test_data.columns) == set(columns)

assert train_user_item["user_id"].notna().all()
assert train_user_item["item_id"].notna().all()

print("Phase 2.4 validation: PASS")
print("Train:", f"{len(train_data):,}")
print("Validation:", f"{len(validation_data):,}")
print("Test:", f"{len(test_data):,}")
print("Train user-item pairs:", f"{len(train_user_item):,}")


Phase 2.4 validation: PASS
Train: 2,356,045
Validation: 200,028
Test: 200,028
Train user-item pairs: 1,939,777


## Phase 2.4 Completion

- Leakage-safe temporal train/validation/test split created.
- Final user-item training representation created.
- All reproducible datasets saved under `data/processed/`.
